In [1]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2
import json
import math

URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [2]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


Number of entities: 151568


In [3]:
for entity in feed.entity[:10]:
    print(entity)

id: "561749tu"
trip_update {
  trip {
    trip_id: "561749"
    start_date: "20260909"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    stop_id: "683048"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 1
    stop_id: "133786"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 2
    stop_id: "42489"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 3
    stop_id: "159883"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 4
    stop_id: "456442"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 5
    stop_id: "376701"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 6
    stop_id: "152836"
    schedule_relationship: SKIPPED
  }
}

id: "325511tu"
trip_update {
  trip {
    trip_id: "325511"
    start_date: "20260908"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {


In [4]:
def parse_trip_updates(
    feed,
    stop_names,
    trip_lines,
    munich_stop_ids
):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.

    stop_names : dict
        Mapping from stop IDs to stop names.

    trip_lines : dict
        Mapping from trip IDs to line names.

    munich_stop_ids : set
        Set of stop IDs located within Munich.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, line, stop, arrival,
        departure, and delay information for Munich stops.
    """

    rows = []

    for entity in feed.entity:

        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        # Get line for this trip
        line = trip_lines.get(
            str(trip.trip_id)
        )

        # Ignore trips that are not part
        # of the selected agencies
        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            stop_id = str(stop.stop_id)

            # Ignore stops outside Munich
            if stop_id not in munich_stop_ids:
                continue

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": stop_id,
                "stop_name": stop_names.get(stop_id),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):

                row["departure_time"] = (
                    datetime.fromtimestamp(
                        stop.departure.time
                    )
                )

                row["departure_delay"] = (
                    stop.departure.delay
                )

            if stop.HasField("arrival"):

                row["arrival_time"] = (
                    datetime.fromtimestamp(
                        stop.arrival.time
                    )
                )

                row["arrival_delay"] = (
                    stop.arrival.delay
                )

            rows.append(row)

    return pd.DataFrame(rows)

In [5]:
def filter_munich_stops(stops_df, munich_map):
    """
    Filter stops to those located inside the Munich polygon.

    Parameters
    ----------
    stops_df : pandas.DataFrame
        Stop data containing stop_lat and stop_lon.

    munich_map : dict
        Munich GeoJSON data.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing only stops inside Munich.
    """

    stops_df = stops_df.copy()

    # Convert WGS84 coordinates to UTM
    stops_df["utm_x"], stops_df["utm_y"] = zip(
        *stops_df.apply(
            lambda row: wgs84_to_utm32(
                row["stop_lat"],
                row["stop_lon"]
            ),
            axis=1
        )
    )

    # Check whether stop is inside Munich
    stops_df["inside_munich"] = stops_df.apply(
        lambda row: point_is_inside_munich(
            row["utm_x"],
            row["utm_y"],
            munich_map
        ),
        axis=1
    )

    # Keep only stops inside Munich
    munich_stops_df = stops_df[
        stops_df["inside_munich"]
    ].copy()

    print(f"Haltestellen insgesamt: {len(stops_df)}")
    print(f"Haltestellen in München: {len(munich_stops_df)}")

    return munich_stops_df

In [6]:
def wgs84_to_utm32(latitude, longitude):
    """
    Convert WGS84 geographic coordinates to UTM Zone 32N.

    Parameters
    ----------
    latitude : float
        Latitude in decimal degrees.

    longitude : float
        Longitude in decimal degrees.

    Returns
    -------
    easting : float
        UTM easting coordinate in meters.

    northing : float
        UTM northing coordinate in meters.
    """

    earth_semi_major_axis = 6378137.0
    eccentricity_squared = 0.00669437999014
    scale_factor = 0.9996

    latitude_radians = math.radians(latitude)
    longitude_radians = math.radians(longitude)

    central_meridian_radians = math.radians(9.0)

    second_eccentricity_squared = (
        eccentricity_squared
        / (1 - eccentricity_squared)
    )

    radius_of_curvature = (
        earth_semi_major_axis
        / math.sqrt(
            1
            - eccentricity_squared
            * math.sin(latitude_radians) ** 2
        )
    )

    tangent_squared = math.tan(latitude_radians) ** 2

    cosine_term = (
        second_eccentricity_squared
        * math.cos(latitude_radians) ** 2
    )

    longitude_difference = (
        math.cos(latitude_radians)
        * (
            longitude_radians
            - central_meridian_radians
        )
    )

    meridional_arc = earth_semi_major_axis * (
        (
            1
            - eccentricity_squared / 4
            - 3 * eccentricity_squared**2 / 64
            - 5 * eccentricity_squared**3 / 256
        )
        * latitude_radians

        - (
            3 * eccentricity_squared / 8
            + 3 * eccentricity_squared**2 / 32
            + 45 * eccentricity_squared**3 / 1024
        )
        * math.sin(2 * latitude_radians)

        + (
            15 * eccentricity_squared**2 / 256
            + 45 * eccentricity_squared**3 / 1024
        )
        * math.sin(4 * latitude_radians)

        - (
            35 * eccentricity_squared**3 / 3072
        )
        * math.sin(6 * latitude_radians)
    )

    easting = (
        scale_factor
        * radius_of_curvature
        * (
            longitude_difference
            + (
                1
                - tangent_squared
                + cosine_term
            )
            * longitude_difference**3
            / 6
            + (
                5
                - 18 * tangent_squared
                + tangent_squared**2
                + 72 * cosine_term
                - 58 * second_eccentricity_squared
            )
            * longitude_difference**5
            / 120
        )
        + 500000.0
    )

    northing = scale_factor * (
        meridional_arc
        + radius_of_curvature
        * math.tan(latitude_radians)
        * (
            longitude_difference**2 / 2

            + (
                5
                - tangent_squared
                + 9 * cosine_term
                + 4 * cosine_term**2
            )
            * longitude_difference**4
            / 24

            + (
                61
                - 58 * tangent_squared
                + tangent_squared**2
                + 600 * cosine_term
                - 330 * second_eccentricity_squared
            )
            * longitude_difference**6
            / 720
        )
    )

    return easting, northing

In [7]:
def preprocess_gtfs(data_dir, munich_agencies):
    """
    Preprocess GTFS static data for selected agencies
    and filter stops to those located within Munich.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.

    munich_stop_ids : set
        Set of stop IDs located within Munich.
    """

    routes_df = pd.read_csv(
        f"{data_dir}/routes.txt"
    )

    trips_df = pd.read_csv(
        f"{data_dir}/trips.txt"
    )

    stops_df = pd.read_csv(
        f"{data_dir}/stops.txt"
    )

    # Convert IDs to strings
    routes_df["route_id"] = (
        routes_df["route_id"].astype(str)
    )

    routes_df["agency_id"] = (
        routes_df["agency_id"].astype(str)
    )

    trips_df["trip_id"] = (
        trips_df["trip_id"].astype(str)
    )

    trips_df["route_id"] = (
        trips_df["route_id"].astype(str)
    )

    stops_df["stop_id"] = (
        stops_df["stop_id"].astype(str)
    )

    # --------------------------------------------------------
    # Filter routes by agency
    # --------------------------------------------------------

    munich_routes = routes_df[
        routes_df["agency_id"].isin(munich_agencies)
    ]

    # --------------------------------------------------------
    # Load Munich GeoJSON
    # --------------------------------------------------------

    geojson_path = f"{data_dir}/munich.geojson"

    with open(geojson_path, "r") as file:
        munich_map = json.load(file)

    # --------------------------------------------------------
    # Filter stops to Munich
    # --------------------------------------------------------

    munich_stops_df = filter_munich_stops(
        stops_df,
        munich_map
    )

    # --------------------------------------------------------
    # Create route mapping
    # --------------------------------------------------------

    route_lines = (
        munich_routes
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    # --------------------------------------------------------
    # Filter trips by route
    # --------------------------------------------------------

    munich_trips = trips_df[
        trips_df["route_id"].isin(route_lines)
    ]

    # --------------------------------------------------------
    # Create trip -> line mapping
    # --------------------------------------------------------

    trip_lines = (
        munich_trips
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .to_dict()
    )

    # --------------------------------------------------------
    # Create stop -> name mapping
    # --------------------------------------------------------

    stop_names = (
        munich_stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    # --------------------------------------------------------
    # Create set of valid Munich stop IDs
    # --------------------------------------------------------

    munich_stop_ids = set(
        munich_stops_df["stop_id"]
        .astype(str)
    )

    return (
        trip_lines,
        stop_names,
        munich_stop_ids
    )

In [8]:
def point_is_inside_polygon(
    point_x,
    point_y,
    polygon_coordinates
):
    """
    Determine whether a point lies inside a polygon.
    """

    point_is_inside = False

    number_of_vertices = len(
        polygon_coordinates
    )

    for vertex_index in range(
        number_of_vertices
    ):

        current_vertex_x, current_vertex_y = (
            polygon_coordinates[vertex_index]
        )

        next_vertex_x, next_vertex_y = (
            polygon_coordinates[
                (vertex_index + 1) % number_of_vertices
            ]
        )

        if (
            current_vertex_y > point_y
        ) != (
            next_vertex_y > point_y
        ):

            x_intersection = (
                (next_vertex_x - current_vertex_x)
                * (point_y - current_vertex_y)
                / (next_vertex_y - current_vertex_y)
                + current_vertex_x
            )

            if point_x < x_intersection:
                point_is_inside = not point_is_inside

    return point_is_inside


def point_is_inside_munich(
    point_x,
    point_y,
    munich_geojson
):
    """
    Determine whether a point lies within Munich.
    """

    for geojson_feature in munich_geojson["features"]:

        geometry = geojson_feature["geometry"]
        geometry_type = geometry["type"]

        if geometry_type == "Polygon":

            polygon_rings = geometry["coordinates"]

            for polygon_ring in polygon_rings:

                if point_is_inside_polygon(
                    point_x,
                    point_y,
                    polygon_ring
                ):
                    return True

        elif geometry_type == "MultiPolygon":

            polygons = geometry["coordinates"]

            for polygon in polygons:

                polygon_rings = polygon

                for polygon_ring in polygon_rings:

                    if point_is_inside_polygon(
                        point_x,
                        point_y,
                        polygon_ring
                    ):
                        return True

    return False



In [ ]:
munich_agencies = ["100", "191", "364"]

trip_lines, stop_names, munich_stop_ids = preprocess_gtfs(
    "../data",
    munich_agencies
)

In [ ]:
df = parse_trip_updates(
    feed,
    stop_names,
    trip_lines,
    munich_stop_ids
)

df.head(100)

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,arrival_time,arrival_delay,departure_time,departure_delay
0,984206,20260905,835,3129,Machtlfinger Straße,6,2026-09-05 14:14:27,27.0,NaT,NaN
1,1091862,20260905,160,212325,Carl-Amery-Platz,14,2026-09-05 14:18:00,0.0,2026-09-05 14:19:00,60.0
2,902239,20260905,19,147434,Paulaner Brauerei,17,2026-09-05 14:45:33,93.0,2026-09-05 14:45:33,93.0
3,902239,20260905,19,494856,Ettalstraße,24,2026-09-05 14:57:33,93.0,2026-09-05 14:57:33,93.0
4,112738,20260905,935,357826,Dankwartstraße,4,2026-09-05 15:05:01,0.0,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...
95,1537243,20260905,59,22613,Freiham Bf (Süd),0,NaT,NaN,2026-09-05 13:37:00,0.0
96,781107,20260905,U4,103672,Bergwachtstraße,9,2026-09-05 14:27:16,196.0,2026-09-05 14:27:28,208.0
97,270373,20260905,220,533576,Blutenburg,17,2026-09-05 14:34:00,0.0,2026-09-05 14:34:00,0.0
98,681988,20260905,16,377351,Mariahilfplatz,2,2026-09-05 15:35:34,214.0,2026-09-05 15:35:35,215.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   trip_id          213 non-null    object        
 1   start_date       213 non-null    object        
 2   line             213 non-null    object        
 3   stop_id          213 non-null    object        
 4   stop_name        213 non-null    object        
 5   stop_sequence    213 non-null    int64         
 6   arrival_time     189 non-null    datetime64[ns]
 7   arrival_delay    189 non-null    float64       
 8   departure_time   192 non-null    datetime64[ns]
 9   departure_delay  192 non-null    float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 16.8+ KB


In [ ]:
def load_existing_realtime_data(
    parquet_path="../data/mvv_realtime.parquet"
):
    """
    Load existing MVV real-time data from a Parquet file.

    Parameters
    ----------
    parquet_path : str
        Path to the existing Parquet file.

    Returns
    -------
    pandas.DataFrame
        Existing MVV real-time data.
    """

    return pd.read_parquet(parquet_path)

In [ ]:
def update_realtime_data(
    existing_df,
    new_df
):
    """
    Add new real-time data and keep the latest
    observation for each trip and stop.

    Parameters
    ----------
    existing_df : pandas.DataFrame
        Previously stored MVV real-time data.

    new_df : pandas.DataFrame
        Newly retrieved MVV real-time data.

    Returns
    -------
    pandas.DataFrame
        Updated MVV real-time data.
    """

    combined_df = pd.concat(
        [
            existing_df,
            new_df
        ],
        ignore_index=True
    )

    combined_df = (
        combined_df
        .drop_duplicates(
            subset=[
                "trip_id",
                "start_date",
                "stop_id"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return combined_df

In [ ]:
def save_realtime_data(
    realtime_df,
    parquet_path="../data/mvv_realtime.parquet"
):
    """
    Save MVV real-time data to a Parquet file.

    Parameters
    ----------
    realtime_df : pandas.DataFrame
        MVV real-time data to save.

    parquet_path : str
        Path where the Parquet file is stored.

    Returns
    -------
    None
        The DataFrame is saved to the specified Parquet file.
    """

    realtime_df.to_parquet(
        parquet_path,
        index=False
    )

In [ ]:
existing_df = load_existing_realtime_data()

updated_df = update_realtime_data(
    existing_df,
    df
)

save_realtime_data(
    updated_df
)

In [ ]:
existing_df = load_existing_realtime_data()


In [ ]:
existing_df

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,1192330,20260904,190,614970,August-Everding-Straße,1,2026-09-05 00:37:33,3.0,2026-09-05 00:37:33,3.0
1,1192330,20260904,190,576765,Sankt Pius,2,2026-09-05 00:37:48,-12.0,2026-09-05 00:37:48,-12.0
2,1192330,20260904,190,686743,Grafinger Straße,3,2026-09-05 00:39:22,-8.0,2026-09-05 00:38:52,-38.0
3,1192330,20260904,190,634241,Altöttinger Straße,4,2026-09-05 00:40:34,4.0,2026-09-05 00:40:11,-19.0
4,1192330,20260904,190,561261,Schlüsselbergstraße,5,2026-09-05 00:41:42,-18.0,2026-09-05 00:41:11,-49.0
...,...,...,...,...,...,...,...,...,...,...
62437,207891,20260905,U6,494287,Hermann-Gmeiner-Weg,12,NaT,NaN,NaT,NaN
62438,1778868,20260905,U2,558604,Aidenbachstraße,2,2026-09-05 15:21:00,0.0,2026-09-05 15:21:00,0.0
62439,99715,20260905,962,389754,Neuaubing Bf./Limesstraße,2,2026-09-05 16:07:00,0.0,2026-09-05 16:07:00,0.0
62440,1502221,20260905,168,130847,Woferlstraße,8,2026-09-05 14:22:52,232.0,2026-09-05 14:22:40,220.0


In [ ]:
existing_df.duplicated(
    subset=[
        "trip_id",
        "start_date",
        "stop_sequence"
    ]
).sum()


np.int64(3)